# Part 1 - 4 Metadata-based NCF-style Models
Metadata-based NCF experiments using only RMSE/MAE.

This notebook uses exactly these metadata features: `beer/style`, `beer/brewerId`, and `beer/ABV`. It runs the four NCF-style experiments and saves the best model choice.

In [ ]:
import os
import json
import copy
import random
from collections import defaultdict
from math import sqrt

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

os.makedirs('./checkpoints', exist_ok=True)

Using device: cuda


In [ ]:
# Step 1: metadata-only experiments with RMSE.
RAW_TRAIN = pd.read_json('./data/beeradvocate_train.json', lines=True)
RAW_VAL = pd.read_json('./data/beeradvocate_val.json', lines=True)
RAW_TEST = pd.read_json('./data/beeradvocate_test.json', lines=True)

user_col = 'review/profileName'
item_col = 'beer/beerId'
rating_col = 'review/overall'
METADATA_CATEGORICAL_COLS = ['beer/style', 'beer/brewerId']
METADATA_NUMERIC_COLS = ['beer/ABV']

required_cols = [user_col, item_col, rating_col] + METADATA_CATEGORICAL_COLS + METADATA_NUMERIC_COLS
for split_name, df in [('train', RAW_TRAIN), ('val', RAW_VAL), ('test', RAW_TEST)]:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {split_name}: {missing}")


def prepare_split(df):
    return df[required_cols].dropna(subset=[user_col, item_col, rating_col]).copy()

train_df = prepare_split(RAW_TRAIN)
val_df = prepare_split(RAW_VAL)
test_df = prepare_split(RAW_TEST)

print('Train:', train_df.shape)
print('Val:', val_df.shape)
print('Test:', test_df.shape)

user2idx = {u: i for i, u in enumerate(train_df[user_col].unique())}
item2idx = {it: i for i, it in enumerate(train_df[item_col].unique())}
unknown_user_idx = len(user2idx)
unknown_item_idx = len(item2idx)
num_users = len(user2idx) + 1
num_items = len(item2idx) + 1


def build_topk_mapping(train_df, column, top_k=100):
    values = train_df[column].fillna('__UNK__').astype(str)
    top_values = values.value_counts().head(top_k).index.tolist()
    mapping = {value: idx for idx, value in enumerate(top_values)}
    unknown_idx = len(mapping)
    return mapping, unknown_idx


categorical_mappings = {}
for col in METADATA_CATEGORICAL_COLS:
    mapping, unknown_idx = build_topk_mapping(train_df, col, top_k=100)
    categorical_mappings[col] = {
        'mapping': mapping,
        'unknown_idx': unknown_idx,
        'num_embeddings': len(mapping) + 1,
    }

numeric_feature_stats = {}
for col in METADATA_NUMERIC_COLS:
    values = pd.to_numeric(train_df[col], errors='coerce')
    mean = float(values.mean()) if values.notna().any() else 0.0
    std = float(values.std()) if values.notna().any() else 1.0
    if std == 0 or np.isnan(std):
        std = 1.0
    numeric_feature_stats[col] = {'mean': mean, 'std': std}


def encode_split(df):
    encoded = df[[user_col, item_col, rating_col] + METADATA_CATEGORICAL_COLS + METADATA_NUMERIC_COLS].copy()
    encoded['user_idx'] = encoded[user_col].map(lambda x: user2idx.get(x, unknown_user_idx))
    encoded['item_idx'] = encoded[item_col].map(lambda x: item2idx.get(x, unknown_item_idx))
    encoded['rating'] = encoded[rating_col].astype(np.float32)

    for col in METADATA_CATEGORICAL_COLS:
        info = categorical_mappings[col]
        encoded[f'{col}__idx'] = encoded[col].fillna('__UNK__').astype(str).map(
            lambda x: info['mapping'].get(x, info['unknown_idx'])
        ).astype(np.int64)

    for col in METADATA_NUMERIC_COLS:
        stats = numeric_feature_stats[col]
        values = pd.to_numeric(encoded[col], errors='coerce').fillna(stats['mean'])
        encoded[f'{col}__num'] = ((values - stats['mean']) / stats['std']).astype(np.float32)

    return encoded[['user_idx', 'item_idx', 'rating'] + [f'{c}__idx' for c in METADATA_CATEGORICAL_COLS] + [f'{c}__num' for c in METADATA_NUMERIC_COLS]]


train_encoded = encode_split(train_df)
val_encoded = encode_split(val_df)
test_encoded = encode_split(test_df)

print('num_users:', num_users)
print('num_items:', num_items)
print('Metadata categorical:', METADATA_CATEGORICAL_COLS)
print('Metadata numeric:', METADATA_NUMERIC_COLS)

Train: (1269292, 6)
Val: (158661, 6)
Test: (158661, 6)
num_users: 27573
num_items: 54912
Metadata categorical: ['beer/style', 'beer/brewerId']
Metadata numeric: ['beer/ABV']


In [ ]:
class MetadataRatingsDataset(Dataset):
    def __init__(self, df, categorical_cols, numeric_cols):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)
        self.categorical_cols = list(categorical_cols)
        self.numeric_cols = list(numeric_cols)
        self.categorical_features = {
            col: torch.tensor(df[f'{col}__idx'].values, dtype=torch.long)
            for col in self.categorical_cols
        }
        self.numeric_features = {
            col: torch.tensor(df[f'{col}__num'].values, dtype=torch.float32)
            for col in self.numeric_cols
        }

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        cat_features = {col: values[idx] for col, values in self.categorical_features.items()}
        num_features = {col: values[idx] for col, values in self.numeric_features.items()}
        return self.users[idx], self.items[idx], self.ratings[idx], cat_features, num_features


def move_feature_dict_to_device(features, device, dtype='long'):
    if dtype == 'long':
        return {k: v.to(device=device, dtype=torch.long) for k, v in features.items()}
    return {k: v.to(device=device, dtype=torch.float32) for k, v in features.items()}


train_dataset = MetadataRatingsDataset(train_encoded, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS)
val_dataset = MetadataRatingsDataset(val_encoded, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS)
test_dataset = MetadataRatingsDataset(test_encoded, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS)


def make_loaders(batch_size):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

In [ ]:
class MetadataMixin:
    def build_metadata_modules(self, embedding_dim, categorical_metadata_info, numeric_metadata_cols):
        self.embedding_dim = embedding_dim
        self.numeric_metadata_cols = list(numeric_metadata_cols or [])
        self.categorical_embeddings = nn.ModuleDict()
        categorical_metadata_info = categorical_metadata_info or {}
        for col, info in categorical_metadata_info.items():
            emb = nn.Embedding(info['num_embeddings'], embedding_dim)
            nn.init.normal_(emb.weight, std=0.01)
            self.categorical_embeddings[col] = emb
        if len(self.numeric_metadata_cols) > 0:
            self.numeric_projection = nn.Linear(len(self.numeric_metadata_cols), embedding_dim)
        else:
            self.numeric_projection = None

    def encode_metadata(self, categorical_features, numeric_features):
        categorical_features = categorical_features or {}
        numeric_features = numeric_features or {}
        metadata_vectors = []
        for col, emb in self.categorical_embeddings.items():
            metadata_vectors.append(emb(categorical_features[col]))
        if self.numeric_projection is not None:
            numeric_stack = torch.stack([numeric_features[col] for col in self.numeric_metadata_cols], dim=1)
            metadata_vectors.append(self.numeric_projection(numeric_stack))
        if metadata_vectors:
            return torch.stack(metadata_vectors, dim=0).mean(dim=0)
        batch_size = next(iter(categorical_features.values())).shape[0] if categorical_features else next(iter(numeric_features.values())).shape[0]
        return torch.zeros(batch_size, self.embedding_dim, device=self.user_embedding.weight.device)


class MetadataGMF(nn.Module, MetadataMixin):
    def __init__(self, num_users, num_items, embedding_dim=64, categorical_metadata_info=None, numeric_metadata_cols=None, **kwargs):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        self.build_metadata_modules(embedding_dim, categorical_metadata_info, numeric_metadata_cols)
        self.output_layer = nn.Linear(embedding_dim * 2, 1)

    def forward(self, user_ids, item_ids, categorical_features=None, numeric_features=None):
        user_vec = self.user_embedding(user_ids)
        item_vec = self.item_embedding(item_ids)
        metadata_vec = self.encode_metadata(categorical_features, numeric_features)
        gmf_out = user_vec * item_vec
        out = self.output_layer(torch.cat([gmf_out, metadata_vec], dim=1)).squeeze(1)
        return out


class MetadataMLP(nn.Module, MetadataMixin):
    def __init__(self, num_users, num_items, embedding_dim=64, hidden_dims=(128, 64, 32), dropout=0.2, categorical_metadata_info=None, numeric_metadata_cols=None, **kwargs):
        super().__init__()
        self.hidden_dims = list(hidden_dims)
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        self.build_metadata_modules(embedding_dim, categorical_metadata_info, numeric_metadata_cols)
        input_dim = embedding_dim * 3
        layers = []
        for h in self.hidden_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(dropout)])
            input_dim = h
        self.mlp = nn.Sequential(*layers)
        self.output_layer = nn.Linear(input_dim, 1)

    def forward(self, user_ids, item_ids, categorical_features=None, numeric_features=None):
        user_vec = self.user_embedding(user_ids)
        item_vec = self.item_embedding(item_ids)
        metadata_vec = self.encode_metadata(categorical_features, numeric_features)
        mlp_in = torch.cat([user_vec, item_vec, metadata_vec], dim=1)
        mlp_out = self.mlp(mlp_in)
        return self.output_layer(mlp_out).squeeze(1)


class MetadataNeuMF(nn.Module, MetadataMixin):
    def __init__(self, num_users, num_items, embedding_dim=64, hidden_dims=(128, 64, 32), dropout=0.2, categorical_metadata_info=None, numeric_metadata_cols=None, **kwargs):
        super().__init__()
        self.hidden_dims = list(hidden_dims)
        self.gmf_user_embedding = nn.Embedding(num_users, embedding_dim)
        self.gmf_item_embedding = nn.Embedding(num_items, embedding_dim)
        self.mlp_user_embedding = nn.Embedding(num_users, embedding_dim)
        self.mlp_item_embedding = nn.Embedding(num_items, embedding_dim)
        for emb in [self.gmf_user_embedding, self.gmf_item_embedding, self.mlp_user_embedding, self.mlp_item_embedding]:
            nn.init.normal_(emb.weight, std=0.01)
        self.user_embedding = self.mlp_user_embedding
        self.build_metadata_modules(embedding_dim, categorical_metadata_info, numeric_metadata_cols)
        input_dim = embedding_dim * 3
        layers = []
        for h in self.hidden_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(dropout)])
            input_dim = h
        self.mlp = nn.Sequential(*layers)
        self.output_layer = nn.Linear(embedding_dim + input_dim, 1)

    def forward(self, user_ids, item_ids, categorical_features=None, numeric_features=None):
        gmf_user = self.gmf_user_embedding(user_ids)
        gmf_item = self.gmf_item_embedding(item_ids)
        gmf_out = gmf_user * gmf_item
        mlp_user = self.mlp_user_embedding(user_ids)
        mlp_item = self.mlp_item_embedding(item_ids)
        metadata_vec = self.encode_metadata(categorical_features, numeric_features)
        mlp_in = torch.cat([mlp_user, mlp_item, metadata_vec], dim=1)
        mlp_out = self.mlp(mlp_in)
        fusion = torch.cat([gmf_out, mlp_out], dim=1)
        return self.output_layer(fusion).squeeze(1)

In [ ]:
HIDDEN_DIMS_CHOICES = {
    '64_32': [64, 32],
    '128_64': [128, 64],
    '128_64_32': [128, 64, 32],
}


def evaluate_rmse(model, data_loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for users, items, ratings, cat_features, num_features in data_loader:
            users = users.to(device)
            items = items.to(device)
            ratings = ratings.to(device)
            cat_features = move_feature_dict_to_device(cat_features, device, dtype='long')
            num_features = move_feature_dict_to_device(num_features, device, dtype='float')
            outputs = torch.clamp(model(users, items, cat_features, num_features), 1.0, 5.0)
            preds.extend(outputs.cpu().numpy())
            targets.extend(ratings.cpu().numpy())
    rmse = sqrt(mean_squared_error(targets, preds))
    mae = mean_absolute_error(targets, preds)
    return rmse, mae


def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3, weight_decay=1e-5, checkpoint_path=None, verbose=1):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_state = copy.deepcopy(model.state_dict())
    best_val_rmse = float('inf')
    history = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        iterator = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False) if verbose else train_loader
        for users, items, ratings, cat_features, num_features in iterator:
            users = users.to(device)
            items = items.to(device)
            ratings = ratings.to(device)
            cat_features = move_feature_dict_to_device(cat_features, device, dtype='long')
            num_features = move_feature_dict_to_device(num_features, device, dtype='float')

            optimizer.zero_grad()
            outputs = model(users, items, cat_features, num_features)
            loss = criterion(outputs, ratings)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_rmse, val_mae = evaluate_rmse(model, val_loader)
        history.append({'epoch': epoch + 1, 'train_loss': total_loss / len(train_loader), 'val_rmse': val_rmse, 'val_mae': val_mae})
        if verbose:
            print(f'Epoch {epoch+1}: Train Loss={total_loss / len(train_loader):.4f}, Val RMSE={val_rmse:.4f}, Val MAE={val_mae:.4f}')
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state = copy.deepcopy(model.state_dict())
            if checkpoint_path:
                torch.save(best_state, checkpoint_path)

    model.load_state_dict(best_state)
    return {'best_val_rmse': best_val_rmse, 'history': history}


def build_model(model_name, params):
    common_kwargs = dict(
        num_users=num_users,
        num_items=num_items,
        embedding_dim=params['embedding_dim'],
        hidden_dims=params['hidden_dims'],
        dropout=params['dropout'],
        categorical_metadata_info=categorical_mappings,
        numeric_metadata_cols=METADATA_NUMERIC_COLS,
    )
    if model_name == 'GMF':
        return MetadataGMF(**common_kwargs).to(device)
    if model_name == 'MLP':
        return MetadataMLP(**common_kwargs).to(device)
    if model_name == 'NeuMF':
        return MetadataNeuMF(**common_kwargs).to(device)
    raise ValueError(f'Unknown model_name: {model_name}')


def initialize_neumf_from_pretrained(neumf_model, gmf_model, mlp_model):
    neumf_model.gmf_user_embedding.weight.data.copy_(gmf_model.user_embedding.weight.data)
    neumf_model.gmf_item_embedding.weight.data.copy_(gmf_model.item_embedding.weight.data)
    neumf_model.mlp_user_embedding.weight.data.copy_(mlp_model.user_embedding.weight.data)
    neumf_model.mlp_item_embedding.weight.data.copy_(mlp_model.item_embedding.weight.data)
    neumf_model.mlp.load_state_dict(copy.deepcopy(mlp_model.mlp.state_dict()))
    gmf_weight = gmf_model.output_layer.weight.data
    mlp_weight = mlp_model.output_layer.weight.data
    if mlp_weight.shape[1] != neumf_model.output_layer.weight.data.shape[1] - gmf_weight.shape[1]:
        # fall back to random fusion head if dimensions differ.
        return neumf_model
    neumf_model.output_layer.weight.data.copy_(0.5 * torch.cat([gmf_weight, mlp_weight], dim=1))
    neumf_model.output_layer.bias.data.zero_()
    return neumf_model


def summarize_experiment(model, val_loader, test_loader):
    val_rmse, val_mae = evaluate_rmse(model, val_loader)
    test_rmse, test_mae = evaluate_rmse(model, test_loader)
    return {'val_rmse': val_rmse, 'val_mae': val_mae, 'test_rmse': test_rmse, 'test_mae': test_mae}


def sample_search_space(trial):
    hidden_dims_key = trial.suggest_categorical('hidden_dims_key', tuple(HIDDEN_DIMS_CHOICES.keys()))
    return {
        'embedding_dim': trial.suggest_categorical('embedding_dim', [32, 64]),
        'hidden_dims': HIDDEN_DIMS_CHOICES[hidden_dims_key],
        'hidden_dims_key': hidden_dims_key,
        'dropout': trial.suggest_float('dropout', 0.1, 0.35),
        'lr': trial.suggest_float('lr', 5e-4, 2e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [256, 512]),
        'epochs': trial.suggest_int('epochs', 6, 10),
    }


def tune_hyperparameters(num_trials=8):
    try:
        import optuna
    except ImportError as exc:
        raise ImportError("Optuna is required for tuning. Install it with `pip install optuna`.") from exc

    def objective(trial):
        params = sample_search_space(trial)
        train_loader, val_loader, _ = make_loaders(params['batch_size'])
        model = build_model('NeuMF', params)
        result = train_model(model, train_loader, val_loader, epochs=params['epochs'], lr=params['lr'], weight_decay=params['weight_decay'], verbose=0)
        trial.set_user_attr('params', params)
        return result['best_val_rmse']

    study = optuna.create_study(direction='minimize', study_name='metadata_ncf_rmse_tuning')
    study.optimize(objective, n_trials=num_trials)
    return study


def run_four_experiments(params):
    train_loader, val_loader, test_loader = make_loaders(params['batch_size'])
    results = {}
    specs = [('MLP', 'MLP'), ('GMF', 'GMF'), ('NeuMF without pretraining', 'NeuMF')]

    for experiment_name, model_name in specs:
        print(f'\n===== Running {experiment_name} =====')
        model = build_model(model_name, params)
        train_model(model, train_loader, val_loader, epochs=params['epochs'], lr=params['lr'], weight_decay=params['weight_decay'], verbose=1)
        results[experiment_name] = {'model_name': model_name, **summarize_experiment(model, val_loader, test_loader)}
        torch.save(model.state_dict(), f"./checkpoints/step1_{experiment_name.replace(' ', '_').replace('/', '_')}.pt")

    print('\n===== Running NeuMF with pretraining =====')
    gmf_pretrained = build_model('GMF', params)
    train_model(gmf_pretrained, train_loader, val_loader, epochs=params['epochs'], lr=params['lr'], weight_decay=params['weight_decay'], verbose=1)
    mlp_pretrained = build_model('MLP', params)
    train_model(mlp_pretrained, train_loader, val_loader, epochs=params['epochs'], lr=params['lr'], weight_decay=params['weight_decay'], verbose=1)
    neumf_pretrained = build_model('NeuMF', params)
    initialize_neumf_from_pretrained(neumf_pretrained, gmf_pretrained, mlp_pretrained)
    train_model(neumf_pretrained, train_loader, val_loader, epochs=params['epochs'], lr=params['lr'], weight_decay=params['weight_decay'], verbose=1, checkpoint_path='./checkpoints/step1_best_model.pt')
    results['NeuMF with pretraining'] = {'model_name': 'NeuMF', 'uses_pretraining': True, **summarize_experiment(neumf_pretrained, val_loader, test_loader)}

    best_name = min(results, key=lambda name: results[name]['val_rmse'])
    payload = {
        'best_experiment_name': best_name,
        'best_model_name': results[best_name]['model_name'],
        'uses_pretraining': results[best_name].get('uses_pretraining', False),
        'best_params': params,
        'metadata_categorical_cols': METADATA_CATEGORICAL_COLS,
        'metadata_numeric_cols': METADATA_NUMERIC_COLS,
        'results': results,
    }
    with open('./checkpoints/step1_best_config.json', 'w') as f:
        json.dump(payload, f, indent=2)
    print('\nSaved step-1 best configuration to ./checkpoints/step1_best_config.json')
    return results, payload

In [ ]:
NUM_TUNING_TRIALS = 8
RUN_HYPERPARAMETER_TUNING = True

if RUN_HYPERPARAMETER_TUNING:
    study = tune_hyperparameters(num_trials=NUM_TUNING_TRIALS)
    best_params = dict(study.best_trial.user_attrs['params'])
    print('Best trial:', study.best_trial.number)
    print('Best validation RMSE:', study.best_value)
else:
    best_params = {
        'embedding_dim': 64,
        'hidden_dims': HIDDEN_DIMS_CHOICES['128_64_32'],
        'hidden_dims_key': '128_64_32',
        'dropout': 0.2,
        'lr': 1e-3,
        'weight_decay': 1e-5,
        'batch_size': 256,
        'epochs': 8,
    }

experiment_results, best_payload = run_four_experiments(best_params)

summary_df = pd.DataFrame(experiment_results).T[['val_rmse', 'val_mae', 'test_rmse', 'test_mae']].sort_values('val_rmse')
summary_df

[I 2026-04-26 18:45:46,503] A new study created in memory with name: metadata_ncf_rmse_tuning
[I 2026-04-26 18:47:32,026] Trial 0 finished with value: 0.5898267466241274 and parameters: {'hidden_dims_key': '128_64_32', 'embedding_dim': 32, 'dropout': 0.1685724241598291, 'lr': 0.0011113517423215087, 'weight_decay': 7.268975706709194e-05, 'batch_size': 512, 'epochs': 8}. Best is trial 0 with value: 0.5898267466241274.
[I 2026-04-26 18:49:16,878] Trial 1 finished with value: 0.5904140741717392 and parameters: {'hidden_dims_key': '128_64_32', 'embedding_dim': 32, 'dropout': 0.1477110566691818, 'lr': 0.0008774606851320473, 'weight_decay': 1.213096430998193e-05, 'batch_size': 512, 'epochs': 8}. Best is trial 0 with value: 0.5898267466241274.
[I 2026-04-26 18:50:49,665] Trial 2 finished with value: 0.59298908469851 and parameters: {'hidden_dims_key': '64_32', 'embedding_dim': 64, 'dropout': 0.26462313440769863, 'lr': 0.001703808508338225, 'weight_decay': 5.3928776181552844e-05, 'batch_size': 

Best trial: 6
Best validation RMSE: 0.5896239379898959

===== Running MLP =====


Epoch 1: Train Loss=0.8097, Val RMSE=0.5957, Val MAE=0.4417


Epoch 2: Train Loss=0.5039, Val RMSE=0.5954, Val MAE=0.4486


Epoch 3: Train Loss=0.4465, Val RMSE=0.5944, Val MAE=0.4413


...

Epoch 5: Train Loss=0.3569, Val RMSE=0.5967, Val MAE=0.4418


Epoch 6: Train Loss=0.3402, Val RMSE=0.5975, Val MAE=0.4419


Epoch 7: Train Loss=0.3279, Val RMSE=0.5989, Val MAE=0.4444


Epoch 8: Train Loss=0.3183, Val RMSE=0.6047, Val MAE=0.4463

Saved step-1 best configuration to ./checkpoints/step1_best_config.json


,val_rmse,val_mae,test_rmse,test_mae
NeuMF without pretraining,0.590864,0.443768,0.57578,0.440279
MLP,0.592895,0.435684,0.579823,0.433334
NeuMF with pretraining,0.593301,0.437864,0.579226,0.432141
GMF,0.608615,0.448178,0.601249,0.456861
